In [ ]:
# ==============================================================================
# 🚀 DO NOT MODIFY: Standardized Notebook Setup
# ==============================================================================
# This cell is designed to work in both Google Colab and local environments.
# It ensures that the environment is correctly configured by cloning (or
# locating) the project repository and installing the necessary dependencies.
#
# ------------------------------------------------------------------------------
#
#  ⚠️  IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY (NOT ON COLAB):
#
#  This cell will automatically find the repository root and configure your
#  environment. Just make sure you have run: pip install -e .[dev]
#
# ------------------------------------------------------------------------------

import os
import subprocess
import sys
from pathlib import Path

# --- Configuration ---
REPO_URL = "https://github.com/BradSegal/ADH-LLM-Tutorials-2025.git"
REPO_DIR = Path("ADH-LLM-Tutorials-2025")  # The name of the directory once cloned
# --- End of Configuration ---


def find_repo_root(start_path: Path) -> Path | None:
    """
    Find the repository root by looking for pyproject.toml.

    Searches upward from start_path until it finds pyproject.toml or hits root.

    Args:
        start_path: Directory to start searching from.

    Returns:
        Path to repository root, or None if not found.
    """
    current = start_path.resolve()
    while current != current.parent:  # Stop at filesystem root
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return None


def detect_active_branch(repo_dir: Path) -> str:
    """
    Determine the active git branch for pulling updates.

    Tries multiple methods to detect the current branch name.

    Args:
        repo_dir: Path to the git repository.

    Returns:
        Branch name (defaults to 'master' if detection fails).
    """
    commands = [
        "git symbolic-ref --short HEAD",
        "git rev-parse --abbrev-ref HEAD",
    ]
    for cmd in commands:
        result = subprocess.run(
            cmd, shell=True, cwd=repo_dir, capture_output=True, text=True
        )
        if result.returncode == 0:
            branch = result.stdout.strip()
            if branch and not branch.startswith("origin/"):
                return branch
    return "master"


def run_cmd(cmd: str, *, cwd: Path | None = None) -> None:
    """
    Run a shell command and raise an error if it fails.

    Args:
        cmd: The command to run.
        cwd: Optional working directory for the command.

    Raises:
        RuntimeError: If the command returns a non-zero exit code.
    """
    result = subprocess.run(cmd, shell=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")


# --- Detect environment ---
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# --- Main setup logic ---
if IN_COLAB:
    print("☁️  Running in Google Colab. Setting up the environment...")

    # Determine repository path
    start_dir = Path.cwd()
    if start_dir.name == REPO_DIR.name:
        repo_path = start_dir
    else:
        repo_path = start_dir / REPO_DIR

    # Clone or update repository
    if not repo_path.exists():
        print(f"📥 Cloning repository from {REPO_URL}...")
        run_cmd(f"git clone --quiet {REPO_URL} {repo_path}")
        print(f"✅ Repository cloned to {repo_path}")
    else:
        print(f"📂 Repository already exists at {repo_path}")
        active_branch = detect_active_branch(repo_path)
        print(f"🔄 Pulling latest changes from branch '{active_branch}'...")
        run_cmd(f"git pull origin {active_branch} --quiet", cwd=repo_path)
        print(f"✅ Repository updated")

    # Verify repository structure
    if not (repo_path / "pyproject.toml").exists():
        raise FileNotFoundError(
            f"Repository structure invalid: pyproject.toml not found in {repo_path}. "
            "The repository may be corrupted."
        )

    # Change working directory and update Python path
    print(f"📁 Changing working directory to {repo_path}")
    os.chdir(repo_path)
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

    # Install dependencies (smart installation - only installs missing packages)
    from core.notebook.setup import smart_install_dependencies

    result = smart_install_dependencies(
        repo_path=repo_path,
        include_dev=True,
        verbose=True,
    )

    # Fail loudly if critical packages failed to install
    if result["failed"]:
        print(f"⚠️  WARNING: {len(result['failed'])} packages failed to install:")
        for pkg in result["failed"]:
            print(f"  - {pkg}")
        print("You may encounter import errors. Please check your internet connection.")

    print("" + "=" * 70)
    print("✅ Environment setup complete! You can now proceed with the notebook.")
    print("=" * 70)

else:
    print("💻 Running in local environment. Configuring...")

    # Find the repository root
    repo_path = find_repo_root(Path.cwd())

    if repo_path is None:
        raise FileNotFoundError(
            "Could not find repository root (no pyproject.toml found). "
            "Please ensure you are running this notebook from within the "
            "ADH-LLM-Tutorials-2025 repository directory."
        )

    print(f"✅ Found repository root: {repo_path}")

    # Change working directory and update Python path
    print(f"📁 Changing working directory to {repo_path}")
    os.chdir(repo_path)

    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

    print("" + "=" * 70)
    print("✅ Local environment configured successfully!")
    print("=" * 70)
    print("⚠️  Please ensure you have run: pip install -e .[dev]")
    print("   (Required for local development)")

# 03 - Sepsis Prediction with LSTM

## Understanding Long Short-Term Memory Networks

In this notebook, we explore the **LSTM (Long Short-Term Memory)** architecture for sepsis prediction. LSTMs are a more advanced type of recurrent neural network that address the limitations of simple RNNs and GRUs through a sophisticated gating mechanism.

### What Makes LSTM Special?

The LSTM architecture introduces three key gates that control information flow:

1. **Forget Gate**: Decides what information from the cell state should be discarded
2. **Input Gate**: Controls what new information should be stored in the cell state
3. **Output Gate**: Determines what parts of the cell state should be output

Unlike the GRU which has a simpler two-gate design, the LSTM maintains a separate **cell state** and **hidden state**. This dual-state mechanism allows LSTMs to capture and maintain long-term dependencies in sequential data more effectively, making them particularly well-suited for medical time-series where early clinical markers might be crucial for predictions hours later.

In [ ]:
# Import required libraries
from pathlib import Path

import matplotlib.pyplot as plt
import yaml

from core.config import LSTMConfig, TrainConfig
from core.data import create_dataloaders
from core.data.physionet_sepsis import get_sepsis_data
from core.models import LSTMModel
from core.notebook import ensure_project_root
from core.train import Trainer

## Step 1: Load Configuration

In [ ]:
project_root = ensure_project_root()
# Load configuration from YAML
config_path = Path("configs/lstm.yaml")
config_dict = yaml.safe_load(config_path.read_text())

# Parse into Pydantic models
model_config = LSTMConfig(**config_dict["model"])
train_config = TrainConfig(**config_dict["training"])

print("Model Configuration:")
print(model_config)
print("\nTraining Configuration:")
print(train_config)

### Understanding the Hyperparameters

Below is a summary of the hyperparameters we'll use to train this LSTM model. These parameters control both the model architecture and the training process.

**🔍 Notice the differences from the GRU notebook:**
- **`hidden_size: 128`** (vs. GRU's 64) - LSTM has more parameters per cell, so we can afford a larger hidden size
- **`batch_size: 64`** (vs. GRU's 32) - Larger batches help stabilize LSTM training
- **`epochs: 10`** (vs. GRU's 20) - LSTM often converges faster due to better gradient flow

In [ ]:
from core.notebook import display_hyperparameter_table

# Display hyperparameters in a formatted table
display_hyperparameter_table(model_config, train_config, model_type="LSTM")

### 🧪 Experimentation Guide

Want to experiment with different hyperparameters? Here's how!

#### **1. Safe Parameters to Modify (Great for Learning)**

**`learning_rate`** - Controls how quickly the model learns
- **Too high** → Unstable training, loss may oscillate or diverge
- **Too low** → Slow convergence, may not reach optimal performance
- **Try:** 0.0001, 0.0005, 0.001 (current), 0.005

**`dropout`** - Prevents overfitting by randomly dropping connections during training
- **Higher values (0.4-0.5)** → More regularization, may underfit
- **Lower values (0.0-0.1)** → Less regularization, may overfit
- **Try:** 0.0, 0.1, 0.2 (current), 0.3, 0.5

**`epochs`** - Number of complete passes through the training data
- LSTM typically converges faster than GRU due to better gradient flow
- **Try:** 5, 10 (current), 15, 20
- **LSTM note:** You might achieve similar performance to GRU in fewer epochs!

#### **2. Architectural Parameters (Moderate Impact)**

**`hidden_size`** - Size of LSTM's cell state and hidden state
- LSTM has **dual states** (cell + hidden), so each unit has more parameters than GRU
- Current: 128 (larger than GRU's 64 because LSTM can handle more capacity efficiently)
- **Try:** 64, 128 (current), 256, 512

**`num_layers`** - Number of stacked LSTM layers
- LSTM's gradient flow is better than simple RNNs, making deeper networks more feasible
- **Try:** 1, 2 (current), 3, 4

#### **3. Comparing LSTM to GRU**

| Parameter | GRU | LSTM | Why Different? |
|-----------|-----|------|----------------|
| `hidden_size` | 64 | 128 | LSTM has better gradient flow, handles larger hidden size |
| `batch_size` | 32 | 64 | Larger batches stabilize LSTM's more complex updates |
| `epochs` | 20 | 10 | LSTM often converges faster |

#### **4. Parameters to Leave Alone**

**`input_size`** (34) - Must match the number of features in the dataset ❌  
**`train_val_split`** (0.8) - Keep consistent for fair model comparison ❌  
**`split_seed`** (42) - Keep at 42 for reproducibility ❌

#### **How to Modify Hyperparameters**

1. **Edit the YAML file:** Open `configs/lstm.yaml` in a text editor
2. **Change the values:** Modify the parameters you want to experiment with
3. **Reload this notebook:** Restart the kernel and run all cells again
4. **Compare results:** Compare with GRU results from notebook 02

#### **Expected Experimental Outcomes**

| Modification | Expected Effect |
|--------------|-----------------|
| Hidden size → 64 | Faster training, performance closer to GRU |
| Hidden size → 256 | Slower training, potentially better long-term dependency capture |
| Epochs → 20 | May overfit; watch for validation loss increasing |
| Batch size → 32 | More updates per epoch, may converge faster but less stable |

#### **Reflection Questions**

Before you start training, think about:
- **Why might LSTM perform better than GRU for sepsis prediction?**  
  *Hint: Think about long-term dependencies in ICU patient trajectories*
  
- **How will you know if the larger `hidden_size` helps or hurts?**  
  *Hint: Compare validation loss and metrics to GRU results*

- **What does it mean if LSTM reaches the same performance as GRU in fewer epochs?**  
  *Hint: Consider computational efficiency and training time*

Let's proceed with training!

## Step 2: Load and Prepare Data

We load the cached PhysioNet dataframe and delegate batching to the `core.data` helper, which caches per-patient tensors and performs a deterministic split.

In [ ]:
# Load the preprocessed PhysioNet dataset
sepsis_df = get_sepsis_data()

num_patients = sepsis_df["patient_id"].nunique()
print(f"Total ICU patient stays: {num_patients:,}")
print(f"Total hourly measurements: {len(sepsis_df):,}")

In [ ]:
# Build deterministic train/validation DataLoaders
train_loader, val_loader = create_dataloaders(
    train_config=train_config,
    df=sepsis_df,
)

print(
    f"Train patients: {len(train_loader.dataset)} | Val patients: {len(val_loader.dataset)}"
)
print(
    f"Using split ratio {train_config.train_val_split:.0%} with seed {train_config.split_seed}"
)

## Step 3: Initialize Model and Trainer

https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html#torch.nn.LSTM

In [ ]:
# Create model
model = LSTMModel(model_config)
print(model)

# Define path to save the best model
model_save_path = Path("models/lstm_best.pt")

# Create trainer with checkpoint saving
trainer = Trainer(
    model, train_loader, val_loader, train_config, save_path=model_save_path
)

### ⚠️ Pre-Training Checkpoint

Before we start training, let's review our configuration one more time. This gives you a final chance to verify everything is set correctly.

In [ ]:
from core.notebook import print_pre_training_summary

# Display pre-training summary
print_pre_training_summary(model, train_loader, val_loader, train_config)

## Step 4: Run Training

In [ ]:
# Train the model and save the best checkpoint
history, best_model_path = trainer.fit()

print(f"\n✅ Training complete! Best model saved to: {best_model_path}")

## Step 5: Visualize Results

In [ ]:
from core.notebook import plot_training_dashboard

# Display comprehensive training dashboard
plot_training_dashboard(history, model_name="LSTM", save_path=str(best_model_path))